# 03 · Subida del dataset auto-etiquetado a Roboflow

>**AVISO: este notebook es documentación del proceso, NO debe re-ejecutarse.**
>
> Las imágenes auto-etiquetadas ya fueron subidas a Roboflow y revisadas manualmente una a una (las 750 imágenes del dataset). Volver a ejecutar este notebook duplicaría las imágenes en el proyecto de Roboflow y podría sobrescribir el trabajo de revisión ya realizado.
>
> Las celdas de código se conservan como evidencia del método empleado, pero se mantienen sin ejecutar.

**Objetivo:** documentar el proceso de subida de las anotaciones automáticas generadas por el modelo v1 a la plataforma Roboflow, donde posteriormente se revisaron y corrigieron manualmente.

**Contexto:** una vez generadas las anotaciones automáticas, se suben a Roboflow para su revisión. Roboflow proporciona una interfaz web especializada para corregir bounding boxes de forma ágil.

**Entrada:** imágenes y anotaciones en `data/auto_labeled/`.

**Salida:** dataset subido a Roboflow para revisión manual.

## 1. Configuración

La API key de Roboflow se introduce de forma segura mediante `getpass` para no dejarla escrita en el notebook.

In [ ]:
from roboflow import Roboflow
from getpass import getpass
from pathlib import Path
import os

REPO_ROOT = Path("../..").resolve()
AUTO_LABELED_DIR = REPO_ROOT / "data" / "auto_labeled"

WORKSPACE = "hugos-workspace-0dwnk"
PROJECT = "football-tactical-analysis"

# API key introducida de forma segura
# api_key = getpass("Introduce tu API key de Roboflow: ")

print(f"Directorio de datos: {AUTO_LABELED_DIR}")
print(f"Workspace: {WORKSPACE}")
print(f"Proyecto: {PROJECT}")

## 2. Conexión con Roboflow

Se establece la conexión con el proyecto de Roboflow mediante la API.

In [ ]:
# rf = Roboflow(api_key=api_key)
# project = rf.workspace(WORKSPACE).project(PROJECT)
# print("Conexión establecida con el proyecto de Roboflow")

## 3. Mapeo de clases

Es importante indicar explícitamente el mapeo entre los índices numéricos de las clases (0, 1, 2) y sus nombres (ball, player, referee). En la primera subida no se especificó este mapeo, lo que provocó que Roboflow creara clases nuevas con nombres numéricos. El problema se resolvió posteriormente con la función Modify Classes de la plataforma. Por ello, aquí se incluye el parámetro `annotation_labelmap` de forma explícita.

In [ ]:
# labelmap = {"0": "ball", "1": "player", "2": "referee"}

# images_dir = AUTO_LABELED_DIR / "images"
# labels_dir = AUTO_LABELED_DIR / "labels"

# images = list(images_dir.glob("*.jpg"))
# print(f"Imágenes a subir: {len(images)}")

## 4. Subida de imágenes con anotaciones

Se recorre cada imagen con su anotación correspondiente y se sube al proyecto de Roboflow, indicando el mapeo de clases correcto.

In [ ]:
# uploaded = 0
# errors = 0

# for img_path in images:
#     label_path = labels_dir / (img_path.stem + ".txt")
#     if not label_path.exists():
#         continue
#     try:
#         project.upload(
#             image_path=str(img_path),
#             annotation_path=str(label_path),
#             annotation_labelmap=labelmap,
#             batch_name="auto-labeled-v3",
#             split="train",
#             num_retry_uploads=3,
#             is_prediction=False
#         )
#         uploaded += 1
#     except Exception as e:
#         errors += 1
#         print(f"Error con {img_path.name}: {e}")

# print(f"\nImágenes subidas: {uploaded}")
# print(f"Errores: {errors}")

## 5. Revisión manual posterior

Tras la subida, se realizó la revisión manual de las 750 imágenes directamente en la interfaz web de Roboflow. La revisión consistió en:

- **Añadir balones no detectados:** el modelo v1 solo detectó 170 balones de los aproximadamente 700 presentes, por lo que se añadieron manualmente los faltantes.
- **Eliminar falsos positivos:** principalmente personas fuera del campo (entrenadores, suplentes, público) detectadas erróneamente.
- **Corregir confusiones de clase:** algunos árbitros detectados como jugadores y viceversa.
- **Reajustar bounding boxes:** cajas que no encerraban correctamente al sujeto.

Este proceso de revisión es el paso clave: el modelo hace el trabajo pesado de proponer detecciones y yo solo corrijo

## 6. Conclusiones

Tras este proceso:

- Las anotaciones automáticas del modelo v1 se subieron a Roboflow.
- Se corrigió el problema inicial de mapeo de clases mediante la función Modify Classes.
- Se revisaron manualmente las 750 imágenes, obteniendo un dataset de alta calidad.
- El dataset resultante queda listo para entrenar el modelo v2, lo que se documenta en el siguiente notebook.

El dataset revisado se aloja en Roboflow, que actúa como copia de seguridad y permite la exportación en el formato requerido por YOLOv8.